# SARIMA 单序列原型

**目的**：对代表性序列跑通 SARIMA 建模流程，验证时间切分、评估接口、网格搜索。

**序列**：
1. (1, GROCERY I) — 主力高销量
2. (1, AUTOMOTIVE) — 稀疏序列

In [ ]:
import sys, os, warnings
sys.path.append(os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.utils import load_time_series, time_split, evaluate_all
from src.models.baseline import SeasonalNaive
from src.models.sarima import SarimaModel

%matplotlib inline
plt.style.use('ggplot')

In [ ]:
s1 = load_time_series(1, 'GROCERY I')
train, val, test = time_split(s1)

fig, ax = plt.subplots(figsize=(14, 4))
train['sales'].plot(ax=ax, label='Train')
val['sales'].plot(ax=ax, label='Validation')
test['sales'].plot(ax=ax, label='Test')
ax.set_title('Store 1, GROCERY I — Sales')
ax.legend()
plt.tight_layout()
plt.savefig('../output/sarima_s1_timeseries.png', dpi=100)
plt.show()

print(f'Train: {train.index.min()} ~ {train.index.max()}  ({len(train)} days)')
print(f'Val:   {val.index.min()} ~ {val.index.max()}    ({len(val)} days)')
print(f'Test:  {test.index.min()} ~ {test.index.max()}  ({len(test)} days)')

In [ ]:
baseline = SeasonalNaive(period=7)
baseline.fit(train['sales'])
preds_seas = baseline.predict(len(val))
metrics_seas = evaluate_all(val['sales'].values, preds_seas)
print('Seasonal Naive Val:', metrics_seas)

In [ ]:
# 用 2016-01 后的数据加速搜索
train_sub = train.loc['2016-01-01':]
print(f'Grid search on {len(train_sub)} days...')

best_params, best_aic = SarimaModel.grid_search(
    train_sub['sales'],
    p_range=[0,1,2], d_range=[0,1], q_range=[0,1,2],
    P_range=[0,1], D_range=[0,1], Q_range=[0,1],
    s=7,
)
print(f'Best params: {best_params}')
print(f'Best AIC: {best_aic:.2f}')

In [ ]:
model = SarimaModel(**best_params)
model.fit(train['sales'])
preds = model.predict(steps=len(val))
metrics_val = evaluate_all(val['sales'].values, preds)
print('SARIMA Val:', metrics_val)

# 对比
comparison = pd.DataFrame({
    'Seasonal Naive': metrics_seas,
    'SARIMA': metrics_val,
})
print('\nComparison:\n', comparison)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
val['sales'].plot(ax=ax, label='Actual', linewidth=2)
pd.Series(preds, index=val.index).plot(ax=ax, label='SARIMA Pred', linestyle='--')
pd.Series(preds_seas, index=val.index).plot(ax=ax, label='Seasonal Naive', linestyle=':')
ax.set_title('SARIMA — Validation Predictions vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig('../output/sarima_s1_prediction.png', dpi=100)
plt.show()

In [ ]:
model.fit(train['sales'])  # refit on all train
preds_test = model.predict(steps=len(test))
metrics_test = evaluate_all(test['sales'].values, preds_test)
print('SARIMA Test:', metrics_test)

In [ ]:
s2 = load_time_series(1, 'AUTOMOTIVE')
t2, v2, te2 = time_split(s2)

params2, aic2 = SarimaModel.grid_search(
    t2.loc['2016-01-01':]['sales'],
    p_range=[0,1], d_range=[0,1], q_range=[0,1],
    P_range=[0,1], D_range=[0,1], Q_range=[0,1],
    s=7,
)
print(f'S2 best params: {params2}, AIC: {aic2:.2f}')

m2 = SarimaModel(**params2)
m2.fit(t2['sales'])
p2 = m2.predict(steps=len(v2))
m2_val = evaluate_all(v2['sales'].values, p2)
print(f'S2 SARIMA Val: {m2_val}')